# Tarea 2 — SOLUCIÓN COMPLETA (Versión Profesor)
### Universidad Cenfotec — Aprendizaje Automático
### Scikit-Learn | Sin PySpark

**Datasets:**
- Parte I: Heart Disease Dataset (clasificación binaria)
- Parte II: Mall Customers Dataset (clustering)

> ⚠️ Este notebook contiene la solución completa. No distribuir a estudiantes.

In [ ]:
# ── Instalación (Google Colab) ────────────────────────────────────────────────
# !pip install scikit-learn matplotlib seaborn scipy pandas numpy -q

import os, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew

# ── Sklearn: preprocesamiento ─────────────────────────────────────────────────
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    train_test_split, cross_val_score, StratifiedKFold
)

# ── Sklearn: clasificación ────────────────────────────────────────────────────
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

# ── Sklearn: clustering ───────────────────────────────────────────────────────
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from scipy.cluster.hierarchy import dendrogram, linkage

# ── Sklearn: métricas ─────────────────────────────────────────────────────────
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, roc_curve, auc,
    confusion_matrix, classification_report
)

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
SEED = 42
np.random.seed(SEED)
print('Entorno listo.')

## 📋 Instrucciones

Completa todas las celdas marcadas con `# TODO` (31 ítems).

**Reglas:** sklearn para todo | sin PySpark | random_state=42 en todos los modelos | `_true_cluster` solo al final | notebook ejecutado sin errores.

**Entrega:** notebook ejecutado (.ipynb) + informe escrito (.pdf).

---

## 0.2. Generar / cargar los datasets

Si tienes los CSV, descomenta la carga. Si no, el bloque siguiente los genera automáticamente.

URL: https://archive.ics.uci.edu/dataset/45/heart+disease
URL: https://www.kaggle.com/datasets/shwetabh123/mall-customers

In [ ]:
# ── Opción A: cargar desde archivo (descomenta si subes el CSV) ───────────────
# from google.colab import files
# uploaded = files.upload()
# df_heart_raw = pd.read_csv('heart_disease.csv')
# df_mall_raw  = pd.read_csv('mall_customers.csv')

print(f'Heart Disease: {df_heart_raw.shape} | target: {df_heart_raw.target.value_counts().to_dict()}')
print(f'Mall Customers: {df_mall_raw.shape}')

---
# PARTE I — CLASIFICACIÓN: HEART DISEASE

## 1. Carga y exploración inicial

In [ ]:
df_heart = df_heart_raw.copy()
print(f'Filas: {df_heart.shape[0]}  Columnas: {df_heart.shape[1]}')
display(df_heart.head(10))
print('\nTipos de datos:')
print(df_heart.dtypes)
print('\nNulos por columna:')
print(df_heart.isnull().sum())

## 2. EDA

In [ ]:
# TODO 2: Calcular estadísticas descriptivas con .describe(include='all').
# Tu código aquí:


In [ ]:
# TODO 3: Graficar distribución del target con barras y porcentajes.
# Tu código aquí:


In [ ]:
# TODO 4: Graficar histogramas de todas las variables en una grilla.
# Tu código aquí:


In [ ]:
# TODO 5: KDE por clase (target=0 vs 1) para variables continuas.
# Marcar la media de cada grupo con línea punteada vertical.
# Tu código aquí:


In [ ]:
# d de Cohen
def cohen_d(df, feature_cols, group_col='target'):
    results = []
    for col in feature_cols:
        g0 = df[df[group_col]==0][col].dropna()
        g1 = df[df[group_col]==1][col].dropna()
        n0, n1 = len(g0), len(g1)
        v0, v1 = g0.var(ddof=1), g1.var(ddof=1)
        std_p = np.sqrt(((n1-1)*v1 + (n0-1)*v0) / (n1+n0-2)) if (v0 and v1) else 1
        d = (g1.mean() - g0.mean()) / std_p if std_p else 0
        abs_d = abs(d)
        efecto = 'Grande' if abs_d>=0.8 else ('Mediano' if abs_d>=0.5 else ('Pequeño' if abs_d>=0.2 else 'Insignificante'))
        results.append({'variable':col,'media_sano':round(g0.mean(),3),
                        'media_enf':round(g1.mean(),3),'d_cohen':round(d,4),'efecto':efecto})
    return pd.DataFrame(results).sort_values('d_cohen', key=abs, ascending=False)

cohen_pd = cohen_d(df_heart, num_cols+['ca','thal','exang','cp','slope'])
print(cohen_pd.to_string(index=False))

colors_d = {'Grande':'tomato','Mediano':'orange','Pequeño':'steelblue','Insignificante':'lightgrey'}
fig, ax = plt.subplots(figsize=(10,5))
ax.barh(cohen_pd['variable'][::-1], cohen_pd['d_cohen'].abs()[::-1],
        color=[colors_d[e] for e in cohen_pd['efecto'][::-1]], edgecolor='black')
for u, ls, lb in [(0.2,'--','pequeño'),(0.5,'-.','mediano'),(0.8,':','grande')]:
    ax.axvline(u, color='black', linestyle=ls, alpha=0.5, label=f'|d|={u} ({lb})')
ax.set_xlabel('d de Cohen (|valor absoluto|)')
ax.set_title('Tamaño del efecto por variable — Heart Disease', fontsize=12)
ax.legend(fontsize=9); plt.tight_layout(); plt.show()

In [ ]:
# TODO 7: Curtosis con .kurtosis(). Clasificar e interpretar.
# Tu código aquí:


In [ ]:
# TODO 8: Mapa de calor de correlaciones de Pearson.
# Tu código aquí:


In [ ]:
# TODO 9: Boxplots de variables numéricas por clase.
# Tu código aquí:


## 3. Tratamiento de valores faltantes

In [ ]:
COLS_MISSING = ['ca', 'thal']

# Estrategia A: imputación con la media (SimpleImputer)
imputer = SimpleImputer(strategy='mean')
df_imputed = df_heart.copy()
df_imputed[COLS_MISSING] = imputer.fit_transform(df_heart[COLS_MISSING])
print(f'Filas tras imputación : {len(df_imputed)} | Nulos restantes: {df_imputed.isnull().sum().sum()}')

# Estrategia B: eliminar filas con nulos
df_dropna = df_heart.dropna(subset=COLS_MISSING)
print(f'Filas tras dropna     : {len(df_dropna)}')
print('\nDecisión: se usa imputación con la media — preserva más datos (303 vs 291).')

## 4. Detección y manejo de outliers (IQR)

In [ ]:
# TODO 12: Método IQR con pandas/numpy. Cuantificar outliers. Boxplots antes/después.
# Tu código aquí:


## 5. Preparación de features y Pipeline

In [ ]:
FEATURE_COLS = [c for c in df_clean.columns if c != 'target']
X = df_clean[FEATURE_COLS].values
y = df_clean['target'].values

# Split estratificado 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y)

print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Distribución train — 0:{(y_train==0).sum()} 1:{(y_train==1).sum()}')
print(f'Distribución test  — 0:{(y_test==0).sum()}  1:{(y_test==1).sum()}')

# Pipeline de preprocesamiento
prep_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler',  StandardScaler())
])
X_train_s = prep_pipeline.fit_transform(X_train)
X_test_s  = prep_pipeline.transform(X_test)
print('\nPipeline (Imputer → StandardScaler) aplicado correctamente.')

## 6. Modelado — Tres algoritmos de clasificación

In [ ]:
# Función de evaluación reutilizable
def evaluar_modelo(nombre, y_true, y_pred, y_prob=None):
    metrics = {
        'Modelo':    nombre,
        'Accuracy':  round(accuracy_score(y_true, y_pred), 4),
        'F1':        round(f1_score(y_true, y_pred, average='macro'), 4),
        'Precision': round(precision_score(y_true, y_pred, average='macro', zero_division=0), 4),
        'Recall':    round(recall_score(y_true, y_pred, average='macro', zero_division=0), 4),
        'AUC-ROC':   round(roc_auc_score(y_true, y_prob[:,1]) if y_prob is not None else 0.0, 4),
    }
    print(f"\n{nombre}")
    for k,v in metrics.items():
        if k != 'Modelo': print(f'  {k:<12}: {v}')
    return metrics

results_clf = {}

### 6.1 Decision Tree Classifier

In [ ]:
# TODO 15: Decision Tree — probar max_depth=3, 5 y 10.
# Usar evaluar_modelo() y guardar en results_clf['DT'].
# Tu código aquí:


In [ ]:
# TODO 16: Visualizar el árbol con plot_tree (max_depth=4).
# Tu código aquí:


### 6.2 Random Forest Classifier

In [ ]:
# TODO 17: Random Forest (n_estimators≥100). Graficar importancia de variables.
# Guardar en results_clf['RF'].
# Tu código aquí:


### 6.3 MLP Classifier (sklearn)

In [ ]:
# TODO 18: MLP — dos arquitecturas (64,32) y (128,64,32).
# Documentar efecto del solver (adam vs sgd) en comentarios.
# Guardar el mejor en results_clf['MLP'].
# Tu código aquí:


## 7. Evaluación y comparación de modelos

In [ ]:
# TODO 19: Tabla comparativa ordenada por AUC-ROC.
# Tu código aquí:


In [ ]:
# TODO 20: Curva ROC del mejor y peor modelo (superpuestas y en paneles separados).
# Marcar el punto óptimo de Youden.
# Tu código aquí:


In [ ]:
# TODO 21: Matriz de confusión del mejor modelo (heatmap seaborn).
# Interpretar: ¿cuál error es más costoso en contexto clínico?
# Tu código aquí:


In [ ]:
# TODO 22: Validación cruzada 5-Fold con cross_val_score.
# Reportar F1 promedio ± std e intervalo. Graficar F1 por fold.
# Tu código aquí:


---
# PARTE II — CLUSTERING: MALL CUSTOMERS

## 8. EDA del dataset de clustering

In [ ]:
df_mall = df_mall_raw.copy()
# _true_cluster se guarda aparte y se elimina del DataFrame de trabajo
true_clusters = df_mall.pop('_true_cluster')
print(f'Dimensiones: {df_mall.shape}')
display(df_mall.head(5))
print('\nNulos:', df_mall.isnull().sum().to_dict())

In [ ]:
SPEND_COLS = ['Age', 'Annual_Income_k', 'Spending_Score']

# Estadísticas descriptivas
display(df_mall[SPEND_COLS].describe().round(3))

# Distribución de Genre
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
genre_vc = df_mall['Genre'].value_counts()
axes[0].bar(genre_vc.index, genre_vc.values, color=['steelblue','tomato'], edgecolor='black')
axes[0].set_title('Distribución por Género')
df_mall[SPEND_COLS].hist(bins=20, ax=axes[1], color='steelblue', edgecolor='black')
plt.suptitle('EDA — Mall Customers', fontsize=12, y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# TODO 25: Codificar Genre con LabelEncoder. Mapa de calor de correlaciones. Skewness.
# Tu código aquí:


In [ ]:
# TODO 26: Outliers IQR sobre las variables de clustering.
# Tu código aquí:


## 9. Preparación de features para clustering

In [ ]:
scaler_mall = StandardScaler()
X_mall = scaler_mall.fit_transform(df_mall_clean[SPEND_COLS])
print(f'Features escaladas: {X_mall.shape}')

# Índice de Hopkins
from sklearn.neighbors import NearestNeighbors
from random import sample as rnd_sample

def hopkins_index(X, m=None, seed=SEED):
    n, d = X.shape
    m = m or int(0.1*n)
    nbrs = NearestNeighbors(n_neighbors=1).fit(X)
    rand_idx = rnd_sample(range(n), m)
    rng_h = np.random.default_rng(seed)
    ujd, wjd = [], []
    for j in range(m):
        u = rng_h.uniform(X.min(axis=0), X.max(axis=0), d).reshape(1,-1)
        ujd.append(nbrs.kneighbors(u, 2, return_distance=True)[0][0][1])
        wjd.append(nbrs.kneighbors(X[[rand_idx[j]]], 2, return_distance=True)[0][0][1])
    H = sum(ujd)/(sum(ujd)+sum(wjd))
    return round(float(H), 4)

H = hopkins_index(X_mall)
print(f'Índice de Hopkins: {H} ({"Apto para clustering" if H>0.7 else "Revisar estructura"})')

## 10. Selección de k — Elbow + Silhouette

In [ ]:
K_RANGE = range(2, 11)
inertias, sil_scores = [], []

for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    labels = km.fit_predict(X_mall)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_mall, labels, metric='euclidean'))
    print(f'  k={k:>2}  Inercia={inertias[-1]:>12,.2f}  Silhouette={sil_scores[-1]:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(list(K_RANGE), inertias, 'o--', color='steelblue', linewidth=2)
axes[0].set_xlabel('k'); axes[0].set_ylabel('Inercia (WCSS)')
axes[0].set_title('Método del Codo'); axes[0].grid(alpha=0.3)

best_k = list(K_RANGE)[int(np.argmax(sil_scores))]
axes[1].plot(list(K_RANGE), sil_scores, 's--', color='tomato', linewidth=2)
axes[1].axvline(best_k, color='darkred', linestyle=':', label=f'Mejor k={best_k}')
axes[1].set_xlabel('k'); axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Análisis de Silhouette'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.suptitle('Selección de k — Mall Customers', fontsize=13, y=1.02)
plt.tight_layout(); plt.show()
print(f'k seleccionado: {best_k}')
K_FINAL = best_k

## 11. Tres algoritmos de clustering

In [ ]:
# TODO 29: Entrenar K-Means, AgglomerativeClustering y GMM con K_FINAL.
# Para Agglomerative: probar linkage=ward/complete/average. Dendrograma.
# Para GMM: graficar BIC vs AIC. Reportar Silhouette de los tres.
# Tu código aquí:


## 12. Visualización con TSNE (PCA previo)

In [ ]:
# TODO 30: PCA (curva de varianza) + TSNE. Graficar 3 paneles comparativos.
# Tu código aquí:


In [ ]:
# COMPLETAR: ver enunciado
# Tu código aquí:


## 13. Perfilado e interpretación de clusters

In [ ]:
# Mejor algoritmo por Silhouette
best_alg = max(results_clust, key=lambda k: results_clust[k]['Silhouette'])
best_labels = {'K-Means': labels_km, 'Agglomerative': labels_agg, 'GMM': labels_gmm}[best_alg]
print(f'Mejor algoritmo: {best_alg}')

df_mall_clean['cluster'] = best_labels

# Perfil en escala original
profile = (df_mall_clean.groupby('cluster')[SPEND_COLS]
           .mean().round(2))
profile['n_clientes'] = df_mall_clean.groupby('cluster').size()
print('\nPerfil de clusters:')
display(profile)

In [ ]:
# COMPLETAR: ver enunciado
# Tu código aquí:
